In [12]:
import pandas as pd
import numpy as np
import sqlite3
import yaml
import os
import sys

project_path = r'C:\Users\VISHNU\Downloads\nifty100_project'
sys.path.append(project_path)
os.chdir(project_path)

# Load financial ratios from database
conn = sqlite3.connect('data/nifty100.db')
ratios = pd.read_sql_query(
    "SELECT * FROM financial_ratios_computed", conn
)
conn.close()

# Load market cap data for P/E, P/B, dividend yield
from src.etl.loader import load_all_data
data       = load_all_data()
market_cap = data['market_cap']
sectors    = data['sectors']

print(f"Ratios: {ratios.shape}")
print(f"Market cap: {market_cap.shape}")

Loading all datasets...

Dataset Summary:
  Dataset                Rows   Cols
  -----------------------------------
  profitandloss          1164     15
  balancesheet           1165     13
  cashflow               1152      7
  companies                92     12
  analysis                 20      6
  documents              1585      4
  prosandcons              16      4
  sectors                  92      6
  market_cap              552      9
  financial_ratios       1184     16
  peer_groups              56      4

All datasets loaded and cleaned successfully!
Ratios: (1159, 43)
Market cap: (552, 9)


In [13]:
# Get latest year per company from ratios
ANALYSIS_YEAR = '2024-03'

latest_ratios = ratios[ratios['year'] == ANALYSIS_YEAR].copy()
print(f"Companies in {ANALYSIS_YEAR}: {len(latest_ratios)}")

# Also get sales from profitandloss for the sales_min filter
pl = data['profitandloss']
latest_pl = pl[pl['year'] == ANALYSIS_YEAR][['company_id', 'sales']]

# Get latest market cap data (2024)
latest_mc = market_cap[market_cap['year'] == 2024][[
    'company_id', 'market_cap_crore', 'pe_ratio',
    'pb_ratio', 'dividend_yield_pct'
]]

# Merge ratios with market cap
screener_df = pd.merge(latest_ratios, latest_mc, on='company_id', how='left')

# Merge with sales
screener_df = pd.merge(screener_df, latest_pl, on='company_id', how='left')

# Merge with sector info
screener_df = pd.merge(
    screener_df,
    sectors[['company_id', 'broad_sector', 'sub_sector']],
    on='company_id', how='left'
)

print(f"Screener DataFrame shape: {screener_df.shape}")
print(f"Sales column present: {'sales' in screener_df.columns}")

Companies in 2024-03: 98
Screener DataFrame shape: (98, 50)
Sales column present: True


In [14]:
def apply_filters(df, filters, broad_sector_col='broad_sector'):
    """
    Applies threshold filters to the screener DataFrame.

    Supported filter suffixes:
        _min  → column must be >= value
        _max  → column must be <= value

    Special rules:
        debt_to_equity_max → skips Financials sector companies
        sales_min          → maps to 'sales' column
    """
    df = df.copy()
    mask = pd.Series([True] * len(df), index=df.index)

    for filter_key, threshold in filters.items():

        # Map sales_min to sales column
        if filter_key == 'sales_min':
            col = 'sales'
            mask &= df[col].fillna(0) >= threshold
            continue

        # Parse column name and direction
        if filter_key.endswith('_min'):
            col = filter_key[:-4]
            direction = 'min'
        elif filter_key.endswith('_max'):
            col = filter_key[:-4]
            direction = 'max'
        else:
            continue

        # Skip if column doesn't exist
        if col not in df.columns:
            print(f"  Warning: column '{col}' not found, skipping filter")
            continue

        # Special rule: skip Financials for D/E filter
        if col == 'debt_to_equity':
            non_financial = df[broad_sector_col] != 'Financials'
            if direction == 'min':
                mask &= (~non_financial) | (df[col].fillna(0) >= threshold)
            else:
                mask &= (~non_financial) | (df[col].fillna(999) <= threshold)
            continue

        # Apply filter
        if direction == 'min':
            mask &= df[col].fillna(-999) >= threshold
        else:
            mask &= df[col].fillna(999) <= threshold

    return df[mask].copy()

# Test with a simple filter
test_result = apply_filters(
    screener_df,
    {'return_on_equity_pct_min': 15.0, 'debt_to_equity_max': 1.0}
)
print(f"Test filter (ROE>15, D/E<1): {len(test_result)} companies")
print(test_result[['company_id', 'return_on_equity_pct',
                    'debt_to_equity']].head(5).to_string(index=False))

Test filter (ROE>15, D/E<1): 48 companies
company_id  return_on_equity_pct  debt_to_equity
       ABB                 32.47            0.02
ADANIPORTS                 15.35            0.94
ADANIPOWER                 48.28            0.80
ASIANPAINT                 29.68            0.13
      ATGL                 18.66            0.43


In [16]:
# Load screener config
with open('config/screener_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

presets = config['presets']

# Run each preset
results = {}
print("Running 6 preset screeners:")
print(f"  {'Preset':<25} {'Companies Found':>16}")
print(f"  {'-'*42}")

for preset_name, preset_config in presets.items():
    filters  = preset_config['filters']
    rank_col = preset_config['rank_by']

    # Apply filters
    filtered = apply_filters(screener_df, filters)

    # Sort by rank column if it exists
    if rank_col in filtered.columns:
        filtered = filtered.sort_values(rank_col, ascending=False)

    results[preset_name] = filtered
    print(f"  {preset_name:<25} {len(filtered):>16} companies")

Running 6 preset screeners:
  Preset                     Companies Found
  ------------------------------------------
  quality_compounder                      21 companies
  value_pick                               7 companies
  growth_accelerator                      19 companies
  dividend_champion                       32 companies
  debt_free_blue_chip                     21 companies
  turnaround_watch                        36 companies


In [ ]:
# Check Quality Compounder results
print("QUALITY COMPOUNDER — Top 10 companies:")
qc = results['quality_compounder']
print(qc[['company_id', 'broad_sector',
          'return_on_equity_pct', 'debt_to_equity',
          'free_cash_flow_cr', 'sales_cagr_5yr']
].head(10).to_string(index=False))

print("\nDEBT FREE BLUE CHIP — Top 10 companies:")
df_bc = results['debt_free_blue_chip']
print(df_bc[['company_id', 'broad_sector',
             'return_on_equity_pct', 'debt_to_equity']
].head(10).to_string(index=False))

print("\nGROWTH ACCELERATOR — Top 10 companies:")
ga = results['growth_accelerator']
print(ga[['company_id', 'broad_sector',
          'net_profit_cagr_5yr', 'sales_cagr_5yr']
].head(10).to_string(index=False))

QUALITY COMPOUNDER — Top 10 companies:
company_id           broad_sector  return_on_equity_pct  debt_to_equity  free_cash_flow_cr  sales_cagr_5yr
ADANIPORTS            Industrials                 15.35            0.94             8250.0           19.58
ADANIPOWER                 Energy                 48.28            0.80            17651.0           16.09
ASIANPAINT              Materials                 29.68            0.13             3556.0           13.03
     CANBK             Financials                 16.72           14.87            13296.0           18.18
   DRREDDY             Healthcare                 19.74            0.07              509.0           12.64
 EICHERMOT Consumer Discretionary                 22.17            0.02              890.0           11.04
   HAVELLS            Industrials                 17.07            0.04              335.0           13.04
   HCLTECH Information Technology                 23.01            0.08            15840.0           12.7

In [17]:
# Verify all 6 presets ran successfully
print("Preset Summary:")
for name, df in results.items():
    status = "✅" if 5 <= len(df) <= 50 else "⚠️  check result count"
    print(f"  {status} {name:<25} → {len(df)} companies")

Preset Summary:
  ✅ quality_compounder        → 21 companies
  ✅ value_pick                → 7 companies
  ✅ growth_accelerator        → 19 companies
  ✅ dividend_champion         → 32 companies
  ✅ debt_free_blue_chip       → 21 companies
  ✅ turnaround_watch          → 36 companies


In [ ]:
# Investigate value_pick — why only 2 companies?
print("VALUE PICK — checking each filter individually:")
print(f"  P/E < 20:          {(screener_df['pe_ratio'] <= 20).sum()} companies")
print(f"  P/B < 3:           {(screener_df['pb_ratio'] <= 3).sum()} companies")
print(f"  D/E < 2:           {(screener_df['debt_to_equity'] <= 2).sum()} companies")
print(f"  Div Yield > 1%:    {(screener_df['dividend_yield_pct'] >= 1).sum()} companies")
print(f"  All combined:      {len(results['value_pick'])} companies")

print()

# Investigate turnaround_watch — why 58 companies?
print("TURNAROUND WATCH — checking each filter individually:")
print(f"  Rev CAGR 3yr > 10%: {(screener_df['sales_cagr_3yr'] >= 10).sum()} companies")
print(f"  FCF > 0:            {(screener_df['free_cash_flow_cr'] >= 0).sum()} companies")
print(f"  Both combined:      {len(results['turnaround_watch'])} companies")

VALUE PICK — checking each filter individually:
  P/E < 20:          15 companies
  P/B < 3:           10 companies
  D/E < 2:           72 companies
  Div Yield > 1%:    74 companies
  All combined:      2 companies

TURNAROUND WATCH — checking each filter individually:
  Rev CAGR 3yr > 10%: 83 companies
  FCF > 0:            71 companies
  Both combined:      58 companies
